# Read-only verified results

This non-canonical CPU utility verifies immutable public Hugging Face artifacts and, when available, replays Drive candidate/OOD/RQ1 packages through the repository validators. It never trains, downloads model weights, repairs evidence, uploads, creates Drive directories, or changes scientific status. Missing and invalid artifacts stay missing or invalid.

Only aggregate metrics are displayed. Raw prompts, generations, blind mappings, and reviewer identities are not printed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from pathlib import Path

MODEL_FAMILY = 'gemma3'  # gemma3 or qwen2_5_vl
DRIVE_ROOTS = {
    'gemma3': Path('/content/drive/MyDrive/em-displacement-vlm'),
    'qwen2_5_vl': Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b'),
}
DRIVE_PROJECT = DRIVE_ROOTS[MODEL_FAMILY]
AUDIT_DRIVE = DRIVE_PROJECT.is_dir()
print('Drive audit enabled:', AUDIT_DRIVE, DRIVE_PROJECT)

In [ ]:
import subprocess

REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
REPO_REF = 'main'
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise SystemExit(f'{REPO_DIR} is not a Git clone; restart Colab.')
    origin = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if origin.rstrip('/') not in {REPO_URL.rstrip('/'), REPO_URL.removesuffix('.git')}:
        raise SystemExit(f'Unexpected origin {origin!r}; restart Colab.')
    dirty = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain=v1', '--untracked-files=all'],
        text=True,
    ).strip()
    if dirty:
        raise SystemExit('Runtime clone is dirty; restart Colab.')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', '--tags', 'origin'])
else:
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)])
target = 'origin/main' if REPO_REF == 'main' else REPO_REF
commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{target}^{{commit}}'], text=True
).strip()
subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', commit])
%cd {REPO_DIR}
print('Auditing with commit:', commit)

In [ ]:
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[dev]'])

In [ ]:
from google.colab import userdata
import os

try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
    print('Loaded HF_TOKEN for authenticated read-only metadata requests.')
else:
    print('No HF_TOKEN: public artifacts will still be checked; private input stays unpinned.')

## Verify and display

The public candidate summaries are fetched at immutable revisions and checked against their registered SHA-256 hashes and adapter LFS hashes. Drive packages are recomputed from bound files. A passed candidate face-sanity review is still not an OOD EM result.

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'scripts'))
from aggregate_rq1 import aggregate_bundles
from em_displacement_vlm.results_audit import (
    audit_drive_artifacts, audit_public_artifacts, build_results_report,
    load_external_registry, render_markdown,
)

registry = load_external_registry(REPO_DIR / 'protocols' / 'external_artifacts.yaml')
public = audit_public_artifacts(registry, token=os.environ.get('HF_TOKEN'))
drive_results = None
if AUDIT_DRIVE:
    drive_results = audit_drive_artifacts(
        DRIVE_PROJECT,
        model_family=MODEL_FAMILY,
        rq1_aggregator=lambda paths: aggregate_bundles(paths, require_protocol=True),
    )
REPORT = build_results_report(public=public, drive=drive_results)
from IPython.display import Markdown, display
display(Markdown(render_markdown(REPORT)))

In [ ]:
import json
print(json.dumps(REPORT, indent=2, sort_keys=True))